In [1]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, gaussian_wind_direction, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_multi_task_gaussians_3D, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# LINEARIZED QUADCOPTER DYNAMICS
A, B, C, D = quad_hover_linear_model()
A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]

In [ ]:
# TRAINING TASKS AND TESTING TASK GENERATION
np.random.seed(5)

# TRAINING TASKS: directions only in allowed quadrants
point_per_task = 50
num_tasks = 6

tasks_all_disturbances = []
tasks_all_x0 = []
tasks_dx = []
tasks_dy = []

training_angles = []  # to store angles for the vector plot
training_means  = []  # amplitudes for vector plot

def sample_training_angle():
    """Sample angle only in quadrants I, II, and IV."""
    while True:
        ang = np.random.uniform(0, 360)
        if (0 <= ang < 180) or (270 <= ang <= 360):
            return ang

print("=== TRAINING TASKS ===")
for i in range(num_tasks):

    # amplitude parameters
    mean  = np.random.uniform(0, 6.0)
    sigma = np.random.uniform(0.1, 0.6)

    # allowed wind direction quadrants
    direction_deg = sample_training_angle()

    # generate disturbances
    dx, dy = gaussian_wind_direction(
        mean=mean,
        sigma=sigma,
        N=point_per_task,
        direction_deg=direction_deg,
        device="cpu"
    )

    dz = -cfg.G * torch.ones_like(dx)
    disturbances = torch.cat([dx, dy, dz], dim=-1)

    tasks_all_disturbances.append(disturbances)
    tasks_dx.append(dx)
    tasks_dy.append(dy)

    x0 = torch.zeros(point_per_task, 12)
    tasks_all_x0.append(x0)

    training_angles.append(direction_deg)
    training_means.append(mean)

    print(f"[Task {i}] mean={mean:.2f}, sigma={sigma:.2f}, direction={direction_deg:.1f}°")

# Visualization of xyz distributions
visualize_multi_task_gaussians_3D(tasks_all_disturbances, filename="results_all_directions/training_dist.png")

# Joint training dataset
tasks_all_disturbances_joint = torch.cat(tasks_all_disturbances, dim=0)
tasks_all_x0_joint = torch.cat(tasks_all_x0, dim=0).float()


# TESTING TASK: direction ONLY in quadrant III (180° to 270°)
print("\n=== TESTING TASK ===")
point_per_task = 50

wind_mean  = np.random.uniform(0, 6.0)
wind_sigma = np.random.uniform(0.1, 0.6)
wind_angle = np.random.uniform(180, 270)   # QUADRANT III ONLY

dx_test, dy_test = gaussian_wind_direction(
    mean=wind_mean,
    sigma=wind_sigma,
    N=point_per_task,
    direction_deg=wind_angle,
    device="cpu"
)

dz_test = -cfg.G * torch.ones_like(dx_test)
disturbances_test = torch.cat([dx_test, dy_test, dz_test], dim=-1)
x0_test = torch.zeros(point_per_task, 12)

testing_angle = wind_angle
testing_mean  = wind_mean

visualize_multi_task_gaussians_3D([disturbances_test], filename="results_all_directions/testing_dist.png")

print(f"[Testing task] mean={wind_mean:.2f}, sigma={wind_sigma:.2f}, direction={wind_angle:.1f}°")

=== TRAINING TASKS ===
[Task 0] mean=1.33, sigma=0.54, direction=74.4°
[Task 1] mean=5.87, sigma=0.27, direction=134.0°
[Task 2] mean=1.43, sigma=0.46, direction=286.6°
[Task 3] mean=0.71, sigma=0.13, direction=321.8°
[Task 4] mean=2.95, sigma=0.29, direction=273.7°
[Task 5] mean=5.15, sigma=0.36, direction=64.8°

=== TESTING TASK ===
[Testing task] mean=5.92, sigma=0.32, direction=259.9°


In [ ]:
# TASKS VISUALIZATION
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch

class HandlerArrow(HandlerPatch):
    def __init__(self, arrowstyle='->', mutation_scale=15, lw=2, **kwargs):
        super().__init__()
        self.arrowstyle = arrowstyle
        self.mutation_scale = mutation_scale
        self.lw = lw

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):

        arrow = FancyArrowPatch(
            (0, height/2), (width, height/2),
            arrowstyle=self.arrowstyle,
            color=orig_handle.get_edgecolor(),
            mutation_scale=self.mutation_scale,
            lw=self.lw,
            joinstyle='miter',
            capstyle='butt',  
        )
        arrow.set_transform(trans)
        return [arrow]

def plot_wind_vectors(training_angles, training_means, testing_angle, testing_mean, filename):

    plt.figure(figsize=(5,5))

    # --- TRAINING VECTORS ---
    for ang, amp in zip(training_angles, training_means):
        x = amp * np.cos(np.deg2rad(ang))
        y = amp * np.sin(np.deg2rad(ang))
        plt.arrow(0, 0, x, y,
                  head_width=0.2, lw=1.5, length_includes_head=True,
                  color="blue", alpha=0.9)

    # --- TESTING VECTOR ---
    xt = testing_mean * np.cos(np.deg2rad(testing_angle))
    yt = testing_mean * np.sin(np.deg2rad(testing_angle))
    plt.arrow(0, 0, xt, yt,
              head_width=0.2, lw=1.5, length_includes_head=True,
              color="red", alpha=0.9)

    # Create dummy arrow handles for legend
    train_arrow = FancyArrowPatch((0,0), (1,0), color="blue")
    test_arrow  = FancyArrowPatch((0,0), (1,0), color="red")

    plt.legend(
        [train_arrow, test_arrow],
        ["Training wind", "Testing wind"],
        handler_map={
            FancyArrowPatch: HandlerArrow(
                arrowstyle='-|>',   # nicer arrow head
                mutation_scale=8,  # bigger arrowhead
                lw=1.5              # thicker line
            )
        },
        loc="upper left"
    )

    # Formatting
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel(r"$d_x$")
    plt.ylabel(r"$d_y$")
    plt.title("Wind Direction Vectors (mean amplitude)")
    plt.grid(True, alpha=0.3, linestyle='-.', zorder=0)
    plt.xlim(-6, 6)
    plt.ylim(-6, 6)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.savefig(filename, dpi=300)
    plt.close()

plot_wind_vectors(training_angles, training_means, testing_angle, testing_mean, "results_all_directions/wind_vectors.png")

In [50]:
# JOINT TRAINING (BASELINE)
device = "cpu"
epochs = 5000
compute_mean_every = 100
lr = 2e-3

joint_policy, train_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=tasks_all_x0_joint, disturbances=tasks_all_disturbances_joint,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True,
    use_linearized=False
)

[ep 100/5000]:   train_loss=3.1394e+05   |    eval_loss=2.8868e+05   |   |x_T|_avg=172.369
[ep 200/5000]:   train_loss=1.6848e+05   |    eval_loss=1.7675e+05   |   |x_T|_avg=137.873
[ep 300/5000]:   train_loss=3.4507e+04   |    eval_loss=2.7740e+04   |   |x_T|_avg=53.475
[ep 400/5000]:   train_loss=7.0670e+03   |    eval_loss=6.9960e+03   |   |x_T|_avg=21.426
[ep 500/5000]:   train_loss=1.6928e+03   |    eval_loss=1.6452e+03   |   |x_T|_avg=4.468
[ep 600/5000]:   train_loss=2.8287e+02   |    eval_loss=2.8237e+02   |   |x_T|_avg=0.688
[ep 700/5000]:   train_loss=2.4990e+03   |    eval_loss=1.6813e+04   |   |x_T|_avg=7.534
[ep 800/5000]:   train_loss=2.1015e+02   |    eval_loss=2.0995e+02   |   |x_T|_avg=0.231
[ep 900/5000]:   train_loss=2.0150e+02   |    eval_loss=2.0131e+02   |   |x_T|_avg=0.203
[ep 1000/5000]:   train_loss=1.9730e+02   |    eval_loss=1.9701e+02   |   |x_T|_avg=0.217
[ep 1100/5000]:   train_loss=1.8841e+02   |    eval_loss=1.8827e+02   |   |x_T|_avg=0.190
[ep 1200/5000

In [ ]:
# PLOT LOSS JOINT TRAINING
plot_loss_evolution(train_means, epochs, compute_mean_every, filename="results_all_directions/loss_joint_training.png")

In [5]:
# META TRAINING
device = "cpu"
epochs = 2000
compute_mean_every = 100
lr_outer_loop = 2e-3            # β
lr_inner_loop = 2e-3            # α
nb_samples_inner_training = 5
nb_samples_inner_testing = 5


meta_policy, logs = meta_train_nn_policy(
    A_d_nt = A_d, B_tilde_d_nt = B_tilde_d,
    init_points_list = tasks_all_x0,          
    disturbances_list = tasks_all_disturbances,
    lr_outer=lr_outer_loop,          
    lr_inner=lr_inner_loop,   
    meta_batch_size=None, 
    K_inner=nb_samples_inner_training,               
    K_outer=nb_samples_inner_testing,    
    epochs=epochs,
    device="cpu",
    verbose_every=compute_mean_every,
    use_linearized=False
)

[MAML] Meta-training on 6 tasks.


/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:389: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_i_prime, (x,))


[ep 100/2000] meta_loss = 7.4499e+05
[ep 200/2000] meta_loss = 4.3824e+05
[ep 300/2000] meta_loss = 1.8900e+05
[ep 400/2000] meta_loss = 1.6173e+05
[ep 500/2000] meta_loss = 1.1323e+05
[ep 600/2000] meta_loss = 6.3219e+04
[ep 700/2000] meta_loss = 3.0991e+04
[ep 800/2000] meta_loss = 6.1809e+03
[ep 900/2000] meta_loss = 4.2407e+03
[ep 1000/2000] meta_loss = 3.1184e+03
[ep 1100/2000] meta_loss = 1.6077e+03
[ep 1200/2000] meta_loss = 8.0111e+02
[ep 1300/2000] meta_loss = 4.8704e+02
[ep 1400/2000] meta_loss = 3.6802e+02
[ep 1500/2000] meta_loss = 2.8613e+02
[ep 1600/2000] meta_loss = 2.3925e+02
[ep 1700/2000] meta_loss = 2.1597e+02
[ep 1800/2000] meta_loss = 2.0050e+02
[ep 1900/2000] meta_loss = 1.9340e+02
[ep 2000/2000] meta_loss = 1.8947e+02


In [ ]:
# PLOT LOSSES META-TRAINING
plot_maml_diagnostics(logs, filename="results_all_directions/loss_meta_training.png")

In [ ]:
# ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
torch.manual_seed(37)

adapted_policy_from_meta_learning, adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta = maml_adapt(
    meta_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=2e-2,
    nb_gradient_descent=6,
    K_samples=5,
    device="cpu",
    use_linearized=False
)

plot_adapted_losses(adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, filename="results_all_directions/meta_adaptation.png")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:38: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacem

In [ ]:
# APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
torch.manual_seed(215)

adapted_policy_from_joint_learning, adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint = maml_adapt(
    joint_policy,
    A_d, B_tilde_d,
    x0_test, disturbances_test,
    lr_inner=4e-2,
    nb_gradient_descent=6,
    K_samples=5,
    device="cpu",
    use_linearized=False
)
plot_adapted_losses(adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, filename="results_all_directions/joint_adaptation.png")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3135411260.py:38: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacem

In [161]:
# COMPARE ADAPTED POLICIES ON TEST TASK
plot_adaptation_comparison(full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta, "results_all_directions/adaptation_comparison.png")

In [174]:
# TESTING ADAPTED POLICY FROM META-LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_meta, U_test_adapted_from_meta = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device),
    linearized=False
)
costs = traj_cost(X_test_adapted_from_meta, U_test_adapted_from_meta)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_meta, filename="results_all_directions/testing_xyz_adapted_from_meta.png")
plot_angles_mean(X_test_adapted_from_meta, filename="results_all_directions/testing_angles_adapted_from_meta.png")
plot_controls_grid_mean(U_test_adapted_from_meta, filename="results_all_directions/testing_controls_adapted_from_meta.png")
animate_trajectories(X_test_adapted_from_meta, "results_all_directions/testing_xyz_3d_adapted_from_meta.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/1201600873.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/1201600873.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/1201600873.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 846.884033203125
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [164]:
# TESTING ADAPTED POLICY FROM JOINT LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(joint_policy, adapted_policy_from_joint_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_joint, U_test_adapted_from_joint = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device),
    linearized=False
)
costs = traj_cost(X_test_adapted_from_joint, U_test_adapted_from_joint)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_joint, filename="results_all_directions/testing_xyz_adapted_from_joint.png")
plot_angles_mean(X_test_adapted_from_joint, filename="results_all_directions/testing_angles_adapted_from_joint.png")
plot_controls_grid_mean(U_test_adapted_from_joint, filename="results_all_directions/testing_controls_adapted_from_joint.png")
animate_trajectories(X_test_adapted_from_joint, "results_all_directions/testing_xyz_3d_adapted_from_joint.mp4")

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3048776779.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3048776779.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_42987/3048776779.py:5: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  re

Adapted test cost = 1841175.375
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200
